In [1]:
from glob import glob
import numpy as np
import torch
from src.sut import YoloSUT

from _analysis import load_jsons, get_class_stats, get_class_flip_stats, extract_class, align_mrm_lists, get_yolo_class_stats, load_img

%load_ext autoreload
%autoreload 2

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth


/home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model_path,

### Load Data and paths

In [2]:
usr_path = "/home/weissl"
mimicry_i_path = f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_class*"
mimicry_i = load_jsons(glob(mimicry_i_path + "/*.json"))
mimicry_i_origins = glob(mimicry_i_path + "/origin*.png")
mimicry_i_targets = glob(mimicry_i_path + "/best*.png")

mimicry_c_path = f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_b*"
mimicry_c = load_jsons(glob(mimicry_c_path + "/*.json"))
mimicry_c_origins = glob(mimicry_c_path + "/origin*.png")
mimicry_c_targets = glob(mimicry_c_path + "/best*.png")

hynea_i_path = f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_1*"
hynea_i = load_jsons(glob(hynea_i_path + "/*.json"))
hynea_i_origins = glob(hynea_i_path + "/origin*.png")
hynea_i_targets = glob(hynea_i_path + "/taget*.png")

hynea_c_path = f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_b*"
hynea_c = load_jsons(glob(hynea_c_path + "/*.json"))
hynea_c_origins = glob(hynea_c_path + "/origin*.png")
hynea_c_targets = glob(hynea_c_path + "/taget*.png")

hynea_y_path = f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_y*"
hynea_y = load_jsons(glob(hynea_y_path + "/*.json"))
hynea_y_origins = glob(hynea_y_path + "/origin*.png")
hynea_y_targets = glob(hynea_y_path + "/taget*.png")

mrm_path = f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_"
mrm_y_path = f"{usr_path}/PycharmProjects/genai_tigs/yolo_sd/test_"
mrm_c_path = f"{usr_path}/PycharmProjects/genai_tigs/sd_weights/celebahq_generatorlow"
mrm_c_origins, mrm_c_targets = align_mrm_lists(glob(mrm_c_path + "/*.png"), glob(mrm_c_path + "/perturb-result/*.png"))
mrm_jsons = load_jsons(glob(mrm_path + "/*/*.json"))
mrm_y_origins, mrm_y_targets = align_mrm_lists(glob(mrm_y_path + "/*.png"), glob(mrm_y_path + "/**/*.npy"))
mrm_c_jsons = load_jsons(glob(mrm_c_path + "/*.json"))

# Calculate Task Performance
### ImageNet

In [3]:
m_o_vals, m_t_vals = mimicry_i["w0_predictions"].values.tolist(), mimicry_i["best_0_y_hat"].values.tolist()
h_o_vals, h_t_vals = hynea_i["y_0"].values.tolist(), hynea_i["y_hat"].values.tolist()
mrm_o_vals, mrm_t_vals = mrm_jsons["initial_logits"].values.tolist(), mrm_jsons["final_logits"].values.tolist()

In [4]:
print("ImageNet Task Performance:")
m_misclass, m_escape = get_class_stats(m_o_vals, m_t_vals)
print(f"\tMimicry Misclass Rate: {m_misclass:.3f}, Escape Ratio: {m_escape:.3f}")

h_misclass, h_escape = get_class_stats(h_o_vals, h_t_vals)
print(f"\tHynea Misclass Rate: {h_misclass:.3f}, Escape Ratio: {h_escape:.3f}")

mrm_misclass, mrm_escape = get_class_stats(mrm_o_vals, mrm_t_vals)
print(f"\tMaryam Misclass Rate: {mrm_misclass:.3f}, Escape Ratio: {mrm_escape:.3f}")

ImageNet Task Performance:
	Mimicry Misclass Rate: 0.820, Escape Ratio: 0.520
	Hynea Misclass Rate: 1.000, Escape Ratio: 0.000
	Maryam Misclass Rate: 1.000, Escape Ratio: 0.842


### CelebA

In [15]:
hynea_c["cl"] = hynea_c["file"].apply(extract_class)
mimicry_c["cl"] = mimicry_c["file"].apply(extract_class)
mrm_c_jsons["cl"] = mrm_c_jsons["file"].apply(lambda x: x.split("_")[-1].split(".")[0])

m_o_vals, m_t_vals = mimicry_c["w0_predictions"].values.tolist(), mimicry_c["best_0_y_hat"].values.tolist()
h_o_vals, h_t_vals = hynea_c["y_0"].values.tolist(), hynea_c["y_hat"].values.tolist()
mrm_o_vals, mrm_t_vals = [e[0] for e in mrm_c_jsons["initial_logits"].values.tolist()], [e[0] for e in mrm_c_jsons["final_logits"].values.tolist()]

print("CelebA Task Performance:")
m_flip, m_sens = get_class_flip_stats(m_o_vals, m_t_vals, mimicry_c["cl"])
print(f"\tMimicry Flip Rate: {m_flip:.3f}, Flip Sensitivity: {m_sens:.3f}")

h_flip, h_sens = get_class_flip_stats(h_o_vals, h_t_vals, hynea_c["cl"])
print(f"\tHynea  Flip Rate: {h_flip:.3f}, Flip Sensitivity: {h_sens:.3f}")

mrm_flip, mrm_sens = get_class_flip_stats(mrm_o_vals, mrm_t_vals, mrm_c_jsons["cl"])
print(f"\tGIFTbench  Flip Rate: {mrm_flip:.3f}, Flip Sensitivity: {mrm_sens:.3f}")

CelebA Task Performance:
	Mimicry Flip Rate: 0.009, Flip Sensitivity: 0.017
	Hynea  Flip Rate: 1.000, Flip Sensitivity: 0.109
	GIFTbench  Flip Rate: 0.467, Flip Sensitivity: 0.079


### Yolo

In [3]:
sut = YoloSUT("yolov8n.pt", device=torch.device("cuda"), return_confidences=True, return_bboxes=False, objectness_exists=False)

In [4]:
h_o_vals, h_t_vals = hynea_y["y_0"].values.tolist(), hynea_y["y_hat"].values.tolist()
g_o_vals = [sut.process_input(torch.tensor(load_img(p).transpose(2,0,1)).unsqueeze(0) / 255.).squeeze().tolist() for p in mrm_y_origins]
g_t_vals = [sut.process_input(torch.tensor(load_img(p).transpose(2,0,1)).unsqueeze(0) / 255.).squeeze().tolist() for p in mrm_y_targets]

y_targets = hynea_y["file"].apply(lambda x: x.split("_")[1]).values.tolist()
mrm_targets = [int(p.split("_")[-1].split(".")[0]) for p in mrm_y_origins]
class_stats = get_yolo_class_stats(h_o_vals, h_t_vals, y_targets)
class_stats_mrm = get_yolo_class_stats(g_o_vals, g_t_vals, mrm_targets)

all_frac, all_frac_mrm = [], []
all_conf, all_conf_mrm = [], []

In [110]:
for cls, stats in class_stats.items():
    all_frac.extend(stats["frac"])
    all_conf.extend(stats["conf_decrease"])

all_frac = np.asarray(all_frac, dtype=float)
all_conf = np.asarray(all_conf, dtype=float)

print(f"\tHynea  Detections Removed: {all_frac.mean():.3f}pm{all_frac.std():.3f}, Confidence Decreased: {all_conf.mean():.3f} pm {all_conf.std():.3f}")

	Hynea  Detections Removed: 1.000pm0.000, Confidence Decreased: 0.947 pm 0.114


In [5]:
for cls, stats in class_stats_mrm.items():
    all_frac_mrm.extend(stats["frac"])
    all_conf_mrm.extend(stats["conf_decrease"])

all_frac_mrm = np.asarray(all_frac_mrm)
all_conf_mrm = np.asarray(all_conf_mrm)

print(f"\tGIFTbench Detections Removed: {all_frac_mrm.mean():.3f}pm{all_frac_mrm.std():.3f}, Confidence Decreased: {all_conf_mrm.mean():.3f} pm {all_conf_mrm.std():.3f}")

	GIFTbench Detections Removed: 0.719pm0.454, Confidence Decreased: 0.831 pm 0.252
